# Double Inverted Pendulum: Deep Q-Network

A complete self-contained JAX/Flax DQN experiment for swing-up and balance of two
serial poles on one cart. Run the notebook top-to-bottom, start with the smoke test,
then tune only the configuration cell. No TIPy source repository is cloned.

MuJoCo stores the second hinge relative to pole 1. The agent observes absolute angles:
`theta1 = q1` and `theta2 = q1 + q2_relative`.


## 1. Prepare Colab
Mount Drive, install headless dependencies, and report the JAX accelerator.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_COLAB or IN_KAGGLE:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade", "-q",
        "jax[cuda12]", "mujoco>=3.2", "gymnasium>=1.0", "flax==0.12.8", "optax==0.2.8",
        "pandas>=2.0", "matplotlib>=3.8", "imageio>=2.34", "imageio-ffmpeg>=0.5",
    ], check=True)

# Keep learning arrays resident on the accelerator and use EGL for GPU-backed video rendering.
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "true")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")

import flax
import jax
import jax.numpy as jnp
import optax

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
print(f"JAX {jax.__version__} | Flax {flax.__version__} | Optax {optax.__version__}")
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())
if (IN_COLAB or IN_KAGGLE) and not gpu_devices:
    raise RuntimeError(
        "GPU accelerator required: enable a GPU in the Colab or Kaggle notebook settings, "
        "restart the runtime, and run all cells again."
    )
if (IN_COLAB or IN_KAGGLE) and flax.__version__ != "0.12.8":
    raise RuntimeError("Restart the notebook runtime, then run all cells again.")

accelerator_probe = jax.jit(lambda x: x @ x)(jnp.ones((64, 64), dtype=jnp.float32))
accelerator_probe.block_until_ready()
probe_device = next(iter(accelerator_probe.devices()))
if (IN_COLAB or IN_KAGGLE) and probe_device.platform != "gpu":
    raise RuntimeError("GPU accelerator required, but the JIT probe ran on " + str(probe_device))
print("JIT accelerator probe:", probe_device)


## 2. Tuning And Persistent Storage
Set `OUTPUT_DIR` to Google Drive. A new run directory is required after changing network or replay dimensions.


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/double/dqn/run-001") if IN_COLAB else Path.cwd() / "double-dqn-run"
CHECKPOINT_DIR, METRICS_PATH = OUTPUT_DIR / "checkpoints", OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH = OUTPUT_DIR / "dashboard.png"

SEED = 42
TOTAL_STEPS = 500_000
MAX_EPISODE_STEPS = 5000
ACTION_LIMIT = 100.0
LEARNING_RATE, GAMMA, BATCH_SIZE = 3e-4, 0.99, 512
BUFFER_CAPACITY, WARMUP_STEPS = 100_000, 2_000
TRAIN_FREQUENCY, TARGET_FREQUENCY = 4, 1000
EPSILON_START, EPSILON_END = 1.0, 0.05
EPSILON_DECAY_STEPS = int(TOTAL_STEPS * 0.75)
CHECKPOINT_EVERY_EPISODES = 25
KEEP_LAST_CHECKPOINTS = 3
SMOKE_TEST = False
if SMOKE_TEST:
    TOTAL_STEPS, MAX_EPISODE_STEPS = 600, 100
    WARMUP_STEPS, BATCH_SIZE, CHECKPOINT_EVERY_EPISODES = 64, 32, 1

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)


## 3. Double-Pendulum MuJoCo Environment

Observation: normalized cart position/velocity, sine and cosine of both absolute angles,
and both absolute angular velocities. The seven actions span -100 N to +100 N. Rail contact
is a true terminal; reaching the time limit is only truncation and still permits Bellman
bootstrapping.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np

MODEL_XML = r"""
<mujoco model="cartpole_double">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4" />
  <default>
    <geom contype="0" conaffinity="0" />
  </default>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>
  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole1_mat" rgba="0.2 0.75 0.3 1" />
    <material name="pole2_mat" rgba="0.2 0.3 0.75 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>
  <worldbody>
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />
    <body name="frame">
      <geom name="tower_left" type="box" pos="-1.3 0 0.4" size="0.1 0.15 0.8" material="metal_mat" />
      <geom name="tower_right" type="box" pos="1.3 0 0.4" size="0.1 0.15 0.8" material="metal_mat" />
      <geom name="rail" type="box" pos="0 0 0.85" size="2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>
    <body name="cart" pos="0 0 1">
      <joint name="cart_slide" type="slide" axis="1 0 0" frictionloss="0" damping="0.1" limited="false" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.15 0.08 0.1" mass="2.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0" axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />
      <body name="pole1" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole1_hinge" type="hinge" axis="0 -1 0" frictionloss="0" damping="0.03" ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.15" mass="0.5" diaginertia="0.00376666666667 0.00381666666667 8.33333333333e-05" />
        <site name="pole1_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole1_geom" type="box" pos="0 0 0.15" size="0.02 0.01 0.15" mass="0.5" material="pole1_mat" />
        <site name="pole1_tip_site" pos="0 0 0.3" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole2_mount_pin" type="capsule" pos="0 0.015 0.3" axisangle="1 0 0 1.5708" size="0.012 0.015" material="metal_mat" />
        <body name="pole2" pos="0 0.03 0.3" quat="1 0 -0 0">
          <joint name="pole2_hinge" type="hinge" axis="0 -1 0" frictionloss="0" damping="0.03" ref="0" limited="false" />
          <inertial pos="0 0 0.15" mass="0.5" diaginertia="0.00376666666667 0.00381666666667 8.33333333333e-05" />
          <site name="pole2_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
          <geom name="pole2_geom" type="box" pos="0 0 0.15" size="0.02 0.01 0.15" mass="0.5" material="pole2_mat" />
          <site name="pole2_tip_site" pos="0 0 0.3" size="0.015" type="sphere" material="site_mat" />
        </body>
      </body>
    </body>
    <camera name="replay" pos="0 6 1.4" fovy="50" xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>
  <sensor>
    <jointpos name="cart_position" joint="cart_slide" />
    <jointpos name="pole1_relative_angle" joint="pole1_hinge" />
    <jointpos name="pole2_relative_angle" joint="pole2_hinge" />
    <jointvel name="cart_velocity" joint="cart_slide" />
    <jointvel name="pole1_relative_velocity" joint="pole1_hinge" />
    <jointvel name="pole2_relative_velocity" joint="pole2_hinge" />
  </sensor>
  <actuator>
    <motor name="cart_motor" joint="cart_slide" gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""

def wrap_angle(value):
    return float((value + np.pi) % (2 * np.pi) - np.pi)

class DoublePendulumEnv(gym.Env):
    def __init__(self, max_episode_steps=5000, action_limit=100.0):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(MODEL_XML)
        self.data = mujoco.MjData(self.model)
        self.max_episode_steps, self.action_limit = max_episode_steps, float(action_limit)
        self.rail_limit, self.dt, self.current_step = 2.0, float(self.model.opt.timestep), 0
        self.action_table = np.linspace(-self.action_limit, self.action_limit, 7)
        self.action_space = spaces.Discrete(7)
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(8,), dtype=np.float32)

    def physical_state(self):
        x, relative1, relative2 = map(float, self.data.qpos)
        dx, relative_speed1, relative_speed2 = map(float, self.data.qvel)
        theta1 = wrap_angle(relative1)
        theta2 = wrap_angle(relative1 + relative2)
        dtheta1 = relative_speed1
        dtheta2 = relative_speed1 + relative_speed2
        return x, theta1, theta2, dx, dtheta1, dtheta2

    def observation(self):
        x, theta1, theta2, dx, dtheta1, dtheta2 = self.physical_state()
        return np.array([np.clip(x / self.rail_limit, -1, 1), np.clip(dx / 5, -1, 1),
                         np.cos(theta1), np.sin(theta1), np.cos(theta2), np.sin(theta2),
                         np.clip(dtheta1 / 20, -1, 1), np.clip(dtheta2 / 20, -1, 1)],
                        dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        mujoco.mj_resetData(self.model, self.data)
        absolute1 = np.pi + self.np_random.uniform(-0.04, 0.04)
        absolute2 = np.pi + self.np_random.uniform(-0.04, 0.04)
        self.data.qpos[:] = [self.np_random.uniform(-0.01, 0.01), absolute1, absolute2 - absolute1]
        absolute_speed1 = self.np_random.uniform(-0.02, 0.02)
        absolute_speed2 = self.np_random.uniform(-0.02, 0.02)
        self.data.qvel[:] = [self.np_random.uniform(-0.01, 0.01), absolute_speed1,
                             absolute_speed2 - absolute_speed1]
        mujoco.mj_forward(self.model, self.data)
        return self.observation(), {}

    def step(self, action):
        self.current_step += 1
        force = float(self.action_table[int(action)])
        self.data.ctrl[0] = force
        mujoco.mj_step(self.model, self.data)
        x, theta1, theta2, dx, dtheta1, dtheta2 = self.physical_state()
        angle_reward = 0.5 * (np.cos(theta1) + np.cos(theta2))
        reward = (angle_reward - 0.25 * (x / self.rail_limit) ** 2 - 0.01 * dx ** 2
                  - 0.003 * (dtheta1 ** 2 + dtheta2 ** 2)
                  - 0.001 * (force / self.action_limit) ** 2
                  + (3.0 if abs(theta1) < 0.35 and abs(theta2) < 0.35 else 0.0))
        terminated = bool(abs(x) >= self.rail_limit)
        if terminated:
            reward -= 100.0
        truncated = bool(self.current_step >= self.max_episode_steps)
        info = {"x": x, "theta1": theta1, "theta2": theta2, "dx": dx,
                "dtheta1": dtheta1, "dtheta2": dtheta2, "force": force}
        return self.observation(), float(reward), terminated, truncated, info

check_env = DoublePendulumEnv(max_episode_steps=10)
first, _ = check_env.reset(seed=7)
second, _ = check_env.reset(seed=7)
assert first.shape == (8,) and np.allclose(first, second)
assert first[2] < -0.95 and first[4] < -0.95
print("Environment check passed:", first)


## 4. Replay Memory
The ring buffer stays on CPU; sampled batches move to JAX.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim, rng):
        self.capacity, self.obs_dim, self.rng = int(capacity), int(obs_dim), rng
        self.obs = np.zeros((capacity, obs_dim), np.float32)
        self.next_obs = np.zeros_like(self.obs)
        self.actions = np.zeros(capacity, np.int32)
        self.rewards = np.zeros(capacity, np.float32)
        self.terminals = np.zeros(capacity, np.float32)
        self.pointer = self.size = 0

    def add(self, obs, action, reward, next_obs, terminal):
        index = self.pointer
        self.obs[index], self.next_obs[index] = obs, next_obs
        self.actions[index], self.rewards[index] = action, reward
        self.terminals[index] = terminal
        self.pointer = (index + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, count):
        indices = self.rng.integers(self.size, size=count)
        return {"obs": self.obs[indices], "next_obs": self.next_obs[indices],
                "actions": self.actions[indices], "rewards": self.rewards[indices],
                "terminals": self.terminals[indices]}

    def state_dict(self):
        return {"capacity": self.capacity, "obs_dim": self.obs_dim, "pointer": self.pointer,
                "size": self.size, "obs": self.obs, "next_obs": self.next_obs,
                "actions": self.actions, "rewards": self.rewards, "terminals": self.terminals}

    def load_state_dict(self, state):
        if (state["capacity"], state["obs_dim"]) != (self.capacity, self.obs_dim):
            raise ValueError("Replay-buffer dimensions changed; use a new run directory.")
        self.pointer, self.size = state["pointer"], state["size"]
        for name in ("obs", "next_obs", "actions", "rewards", "terminals"):
            getattr(self, name)[:] = state[name]


## 5. Q-Network
The larger 8D coupled state uses two 256-unit hidden layers.


In [ ]:
import flax.linen as nn
import jax.numpy as jnp
import optax

class QNetwork(nn.Module):
    actions: int

    @nn.compact
    def __call__(self, observation):
        hidden = nn.relu(nn.Dense(256)(observation))
        hidden = nn.relu(nn.Dense(256)(hidden))
        return nn.Dense(self.actions)(hidden)


## 6. DQN Agent
Only rail termination masks the target; time-limit truncation keeps bootstrapping.


In [ ]:
class DQNAgent:
    def __init__(self, obs_dim, actions, seed):
        self.actions, self.rng = int(actions), np.random.default_rng(seed)
        self.total_steps, self.epsilon = 0, EPSILON_START
        self.network = QNetwork(actions)
        self.online = self.network.init(jax.random.PRNGKey(seed), jnp.zeros((1, obs_dim)))
        self.target = self.online
        self.optimizer = optax.chain(optax.clip_by_global_norm(10.0), optax.adam(LEARNING_RATE))
        self.optimizer_state = self.optimizer.init(self.online)
        self.buffer = ReplayBuffer(BUFFER_CAPACITY, obs_dim, self.rng)
        self.predict = jax.jit(lambda parameters, obs: self.network.apply(parameters, obs))
        self.update_jit = jax.jit(self._update)

    def _update(self, online, target, optimizer_state, batch):
        def loss_function(parameters):
            q_values = self.network.apply(parameters, batch["obs"])
            selected = q_values[jnp.arange(batch["actions"].shape[0]), batch["actions"]]
            next_values = jnp.max(self.network.apply(target, batch["next_obs"]), axis=-1)
            targets = batch["rewards"] + GAMMA * next_values * (1.0 - batch["terminals"])
            return jnp.mean((selected - jax.lax.stop_gradient(targets)) ** 2)
        loss, gradients = jax.value_and_grad(loss_function)(online)
        updates, optimizer_state = self.optimizer.update(gradients, optimizer_state, online)
        return optax.apply_updates(online, updates), optimizer_state, loss

    def act(self, observation, greedy=False):
        if not greedy and self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.actions))
        values = np.asarray(self.predict(self.online, jnp.asarray(observation)[None]))[0]
        return int(self.rng.choice(np.flatnonzero(values == values.max())))

    def observe(self, obs, action, reward, next_obs, terminal):
        self.buffer.add(obs, action, reward, next_obs, terminal)
        self.total_steps += 1
        fraction = min(1.0, self.total_steps / max(1, EPSILON_DECAY_STEPS))
        self.epsilon = EPSILON_START + fraction * (EPSILON_END - EPSILON_START)
        loss = None
        if (self.total_steps >= WARMUP_STEPS and self.total_steps % TRAIN_FREQUENCY == 0
                and self.buffer.size >= BATCH_SIZE):
            batch = jax.tree.map(jnp.asarray, self.buffer.sample(BATCH_SIZE))
            self.online, self.optimizer_state, loss = self.update_jit(
                self.online, self.target, self.optimizer_state, batch)
            loss = float(loss)
        if self.total_steps % TARGET_FREQUENCY == 0:
            self.target = self.online
        return loss

    def state_dict(self):
        return {"online": jax.device_get(self.online), "target": jax.device_get(self.target),
                "optimizer_state": jax.device_get(self.optimizer_state),
                "total_steps": self.total_steps, "epsilon": self.epsilon,
                "rng": self.rng.bit_generator.state, "buffer": self.buffer.state_dict()}

    def load_state_dict(self, state):
        self.online = jax.tree.map(jnp.asarray, state["online"])
        self.target = jax.tree.map(jnp.asarray, state["target"])
        self.optimizer_state = jax.tree.map(jnp.asarray, state["optimizer_state"])
        self.total_steps, self.epsilon = state["total_steps"], state["epsilon"]
        self.rng.bit_generator.state = state["rng"]
        self.buffer.load_state_dict(state["buffer"])


## 7. Atomic Checkpoints And Metrics
Resume restores networks, optimizer, replay memory, epsilon, counters, and random state.


In [ ]:
import csv
from datetime import datetime, timezone
import os
import pickle
import time

METRIC_FIELDS = ["timestamp", "episode", "total_steps", "reward", "loss", "epsilon",
                 "episode_length", "steps_per_second", "both_upright"]

def save_checkpoint(agent, episode):
    payload = {"version": 1, "episode": int(episode), "agent": agent.state_dict()}
    path = CHECKPOINT_DIR / f"checkpoint_{agent.total_steps:012d}.pkl"
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as file:
        pickle.dump(payload, file, pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    temporary.replace(path)
    for old in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"))[:-KEEP_LAST_CHECKPOINTS]:
        old.unlink()
    return path

def restore_checkpoint(agent):
    for path in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"), reverse=True):
        try:
            with path.open("rb") as file:
                payload = pickle.load(file)
            if payload.get("version") != 1:
                raise ValueError("Unsupported checkpoint version")
            agent.load_state_dict(payload["agent"])
            print("Resumed", path.name)
            return int(payload["episode"])
        except Exception as error:
            print("Skipped", path.name, error)
    return 0

def append_metrics(row):
    new_file = not METRICS_PATH.exists()
    with METRICS_PATH.open("a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=METRIC_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)
        file.flush()
        os.fsync(file.fileno())


## 8. Train Or Resume
The first update compiles JAX and will be slower. Checkpointing occurs at episode boundaries.


In [ ]:
environment = DoublePendulumEnv(MAX_EPISODE_STEPS, ACTION_LIMIT)
agent = DQNAgent(environment.observation_space.shape[0], environment.action_space.n, SEED)
episode = restore_checkpoint(agent)
session_start, session_start_steps = time.perf_counter(), agent.total_steps

try:
    while agent.total_steps < TOTAL_STEPS:
        episode += 1
        observation, _ = environment.reset(seed=SEED + episode)
        episode_reward, losses, both_upright = 0.0, [], False
        for episode_length in range(1, MAX_EPISODE_STEPS + 1):
            action = agent.act(observation)
            next_observation, reward, terminated, truncated, info = environment.step(action)
            loss = agent.observe(observation, action, reward, next_observation, terminated)
            if loss is not None:
                losses.append(loss)
            observation = next_observation
            episode_reward += reward
            both_upright |= abs(info["theta1"]) < 0.35 and abs(info["theta2"]) < 0.35
            if terminated or truncated or agent.total_steps >= TOTAL_STEPS:
                break
        elapsed = max(time.perf_counter() - session_start, 1e-9)
        append_metrics({"timestamp": datetime.now(timezone.utc).isoformat(), "episode": episode,
                        "total_steps": agent.total_steps, "reward": episode_reward,
                        "loss": np.mean(losses) if losses else np.nan, "epsilon": agent.epsilon,
                        "episode_length": episode_length,
                        "steps_per_second": (agent.total_steps - session_start_steps) / elapsed,
                        "both_upright": int(both_upright)})
        if episode % CHECKPOINT_EVERY_EPISODES == 0:
            print("Saved", save_checkpoint(agent, episode))
        if episode % 10 == 0:
            print(episode, agent.total_steps, round(episode_reward, 1), round(agent.epsilon, 3))
finally:
    save_checkpoint(agent, episode)


## 9. Static Training Dashboard
The success trace records whether both poles entered the upright region during an episode.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(METRICS_PATH).drop_duplicates("episode", keep="last").sort_values("episode")
window = min(20, len(metrics))
figure, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
figure.suptitle("TIPy Double Pendulum DQN", fontsize=16, fontweight="bold")
axes[0, 0].plot(metrics.episode, metrics.reward, alpha=0.3)
axes[0, 0].plot(metrics.episode, metrics.reward.rolling(window, min_periods=1).mean())
axes[0, 0].set_title("Reward and moving average")
axes[0, 1].plot(metrics.episode, metrics.loss); axes[0, 1].set_title("TD loss")
axes[0, 2].plot(metrics.episode, metrics.epsilon); axes[0, 2].set_title("Epsilon")
axes[1, 0].plot(metrics.episode, metrics.episode_length); axes[1, 0].set_title("Episode length")
axes[1, 1].plot(metrics.episode, metrics.steps_per_second); axes[1, 1].set_title("Steps/second")
axes[1, 2].plot(metrics.episode, metrics.both_upright); axes[1, 2].set_title("Both upright reached")
for axis in axes.flat:
    axis.set_xlabel("Episode")
    axis.grid(alpha=0.25)
figure.savefig(DASHBOARD_PATH, dpi=160)
plt.show()
print("Saved", DASHBOARD_PATH)


## 10. Greedy Evaluation
Evaluation disables exploration and performs no learning.


In [ ]:
evaluation_rewards, captures = [], 0
for evaluation_episode in range(10):
    observation, _ = environment.reset(seed=40_000 + evaluation_episode)
    total_reward, captured = 0.0, False
    while True:
        action = agent.act(observation, greedy=True)
        observation, reward, terminated, truncated, info = environment.step(action)
        total_reward += reward
        captured |= abs(info["theta1"]) < 0.35 and abs(info["theta2"]) < 0.35
        if terminated or truncated:
            break
    evaluation_rewards.append(total_reward)
    captures += int(captured)
print("Greedy reward mean/std:", np.mean(evaluation_rewards), np.std(evaluation_rewards))
print("Both-pole capture rate:", captures / len(evaluation_rewards))


## Replay The Trained Run

Colab cannot reliably open MuJoCo's interactive desktop viewer. This block runs a
deterministic evaluation, streams rendered frames directly into an MP4, saves it in
the run directory, and displays it inline. It replays the current trained policy or
controller; it is not an exact recording of a stochastic training episode.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED, REPLAY_SECONDS, REPLAY_FPS = 50_004, 10.0, 50
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"
replay_env = DoublePendulumEnv(int(REPLAY_SECONDS / 0.005), ACTION_LIMIT)
observation, _ = replay_env.reset(seed=REPLAY_SEED)
renderer = mujoco.Renderer(replay_env.model, height=480, width=640)
writer = imageio.get_writer(REPLAY_PATH, fps=REPLAY_FPS, codec="libx264", quality=8)
frame_stride = max(1, round(1 / (replay_env.dt * REPLAY_FPS)))
try:
    for step in range(replay_env.max_episode_steps):
        observation, _, terminated, truncated, _ = replay_env.step(agent.act(observation, greedy=True))
        if step % frame_stride == 0:
            renderer.update_scene(replay_env.data, camera="replay")
            writer.append_data(renderer.render())
        if terminated or truncated:
            break
finally:
    writer.close()
    renderer.close()
print("Saved replay:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
